# Artefactual Package Demo: Hallucination Detection with WEPR 

This notebook demonstrates the `artefactual` package for scoring LLM outputs, specifically focusing on hallucination detection using entropy-based methods. Here we will use WEPR (Weighted Entropy Production Rate) learned weights per rank, with certain ranks being more informative than others.

We explore two examples:
1.  **General Knowledge Question**: 

    *"What is the capital city of France?"* → `"Paris."` (expected high certainty, low entropy expected)

2.  **Hallucination Trigger**: Asking about the first author of our paper (Charles Moslonka) to observe how the model hallucinates biographical details.

    *"Who is Charles Moslonka?"* → a fabricated biography (expected high uncertainty, high entropy expected)

We will use:
* **JSON fixture (open_ai_responses_top15.json):** two mock OpenAI Responses API outputs with **15 top logprobs per token**, matching the rank count the WEPR weights were trained on
* **Published detector:** named by its own Hugging Face repository (`artefactory/wepr-falcon3`), fetched on first use and cached. WEPR coefficients are fixed at the rank count they were trained at, so `k` must match — passing a different value raises rather than producing a mis-shaped score.
* **WEPR:** scorer from the artefactual package via the scikit-learn pipeline API
* **Visualizations:** each token highlighted by its own score, so the uncertain stretches of an answer are visible rather than inferred.


In [1]:
# On Colab, uncomment to install the package and fetch the files this notebook reads.
# !pip install -q artefactual
# !wget -q https://raw.githubusercontent.com/artefactory/artefactual/main/docs/examples/open_ai_responses_top15.json

In [2]:
import json
from pathlib import Path

from IPython.display import HTML, display

from artefactual.scoring import WEPR

In [3]:
# The Hugging Face repository holding the published WEPR detector for the model that
# produced these responses. A path to your own `.skops` file works too.
DETECTOR = "artefactory/wepr-falcon3"
DATA_PATH = "open_ai_responses_top15.json"

# The rank count the detector was calibrated at, which the fixture also carries. Passing a
# different value raises rather than producing a mis-shaped score, and a response narrower
# than K is refused rather than padded: the missing ranks are unfetched, not absent, so
# filling them would understate the entropy.
K = 15

# Where the highlighting below changes colour.
THRESHOLD_LOW = 0.35  # at or below -> green, the model was confident here
THRESHOLD_HIGH = 0.70  # above -> red, it was not

## Load Example Responses

The fixture contains two responses, to illustrate the contrast between a certain and an
uncertain answer.

In [4]:
with Path(DATA_PATH).open(encoding="utf-8") as f:
    data = json.load(f)

responses = data["responses"]
print(f"Loaded {len(responses)} responses")

Loaded 2 responses


## Build the WEPR Pipeline

The detector is fetched from the Hub on first use, then cached. `WEPR.from_pretrained`
always requires a repository id or a path. The raw OpenAI Responses API dicts go straight
to the pipeline; parsing is its first step.

In [5]:
detector = WEPR.from_pretrained(DETECTOR, k=K)

/Users/hicham.randrianarivo/Work/artefactual/.workspaces/demos/.venv/lib/python3.13/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## Sequence-Level Scoring

`predict_proba(response)` returns an array of shape `(n_sequences, 2)`. Column 1 is the
hallucination probability: higher means the model was more uncertain while generating the
answer. Nothing here was checked against any source.

In [6]:
for resp in responses:
    prompt = resp["metadata"]["prompt"]
    text = resp["output"][0]["content"][0]["text"]
    score = detector.predict_proba(resp)[0, 1]

    print(f"Prompt : {prompt}")
    print(f"Answer : {text}")
    print(f"Score  : {score:.2f}")
    print()

Prompt : What is the capital city of France? Please answer briefly.
Answer : Paris.
Score  : 0.08

Prompt : Who is Charles Moslonka ? Where was he born ? Please answer in two sentences.
Answer : Charles Moslonka is a French singer born in Lyon in 1985.
Score  : 0.99



## Token-Level Scoring

`predict_token_proba(response)` returns `(n_sequences, max_tokens, 1)`: the same
probability, per token, which is what says *where* in the answer the model was uncertain.


* **`n_sequences`**: response index.
* **`max_tokens`**: token index within the sequence.
* **`1`**: the per-token scalar hallucination probability.

In [7]:
def color_for(score) -> str:
    """Green below THRESHOLD_LOW, red above THRESHOLD_HIGH, yellow in between."""
    if score <= THRESHOLD_LOW:
        return "rgba(0, 255, 0, 0.3)"
    if score <= THRESHOLD_HIGH:
        return "rgba(255, 255, 0, 0.3)"
    return "rgba(255, 0, 0, 0.3)"


def highlight(tokens, scores) -> HTML:
    """The answer as it was generated, each token on a background reading its own score."""
    spans = []
    for token, score in zip(tokens, scores, strict=True):
        shown = token.replace("\n", "<br>")
        spans.append(
            f'<span style="background-color: {color_for(score)}; padding: 2px; margin: 1px; '
            f'border-radius: 3px;">{shown}</span>'
        )
    return HTML('<div style="font-family: monospace; font-size: 14px; line-height: 1.5;">' + "".join(spans) + "</div>")

In [8]:
for resp in responses:
    prompt = resp["metadata"]["prompt"]
    tokens = [t["token"] for t in resp["output"][0]["content"][0]["logprobs"]]
    scores = detector.predict_token_proba(resp)[0, :, 0]

    print(f"Prompt: {prompt}")
    display(highlight(tokens, scores))

Prompt: What is the capital city of France? Please answer briefly.


Prompt: Who is Charles Moslonka ? Where was he born ? Please answer in two sentences.


## Your model is not one of the published detectors?

A detector reads one model's confidence, so the four published ones only score the four
models they were trained on. `WEPR(k=15).fit(responses, y)` fits your own, on that
model's answers and a verdict on each.